# E-Commerce Customer Churn Prediction Using Machine Learning

**Academic Mini Project — Artificial Intelligence & Data Science**

## Abstract
This project uses historical online retail transactions to construct customer-level behavioral features and predict whether a customer makes no purchase during a defined future 90-day window. Logistic Regression, Random Forest, and XGBoost are compared using accuracy, precision, recall, F1-score, and ROC-AUC. A Random Forest model is saved for use in the Streamlit application.

## Problem statement
Customer churn can reduce repeat sales and customer lifetime value. The objective of this project is to use past purchase behavior to classify customers according to whether they make any purchase in the following 90-day observation window.

## Important interpretation
The data is historical (2010–2011). The churn label is defined relative to a fixed snapshot date of **1 September 2011** and a 90-day future window. Therefore, this is an academic demonstration of the method, not a model validated on current retail data.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

ValueError: mount failed

## 1. Import libraries and load the dataset

Upload `Online Retail.xlsx` to the Colab session before running the loading cell. The original dataset contains 541,909 transaction rows and 8 columns.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

print("Libraries imported successfully!")

In [ ]:
file_path = "/content/Online Retail.xlsx"
df = pd.read_excel(file_path)

print("Dataset loaded successfully!")
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])
df.head()

## 2. Initial data inspection

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df.isnull().sum()

In [ ]:
print("Duplicate rows:", df.duplicated().sum())

## 3. Data cleaning
Remove rows without customer IDs, cancelled invoices, non-positive quantities, and non-positive prices. Convert customer IDs and invoice dates to appropriate data types.

In [ ]:
# Remove rows without CustomerID
df = df.dropna(subset=['CustomerID'])

# Remove cancelled transactions
df = df[~df['InvoiceNo'].astype(str).str.startswith('C')]

# Remove invalid quantities and prices
df = df[df['Quantity'] > 0]
df = df[df['UnitPrice'] > 0]

# Convert CustomerID to integer
df['CustomerID'] = df['CustomerID'].astype(int)

# Convert InvoiceDate to datetime
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

print("Data cleaned successfully!")
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

In [ ]:
print("Minimum date:", df['InvoiceDate'].min())
print("Maximum date:", df['InvoiceDate'].max())

## 4. Define the observation windows

- **History period:** transactions before 1 September 2011.
- **Future period:** transactions from 1 September 2011 up to (but not including) 30 November 2011.
- **Churn = 1:** no purchase in the future window.
- **Churn = 0:** at least one purchase in the future window.

The future-period transactions are used only to create the target label, not as model input features.

In [ ]:
# Define the snapshot date
snapshot_date = pd.Timestamp('2011-09-01')

# 90-day future period
future_end = snapshot_date + pd.Timedelta(days=90)

print("Snapshot date:", snapshot_date)
print("Future period ends:", future_end)

In [ ]:
# Transactions used to understand past customer behavior
history = df[df['InvoiceDate'] < snapshot_date].copy()

# Transactions used only to determine whether the customer churned
future = df[
    (df['InvoiceDate'] >= snapshot_date) &
    (df['InvoiceDate'] < future_end)
].copy()

print("History transactions:", len(history))
print("Future transactions:", len(future))

## 5. Customer-level feature engineering
Create six features from historical behavior: Recency, Frequency, Monetary, Total Quantity, Unique Products, and Active Days.

In [ ]:
# Create total amount for each transaction
history['TotalAmount'] = history['Quantity'] * history['UnitPrice']

# Create customer-level features
customer_features = history.groupby('CustomerID').agg(
    Recency=('InvoiceDate', lambda x: (snapshot_date - x.max()).days),
    Frequency=('InvoiceNo', 'nunique'),
    Monetary=('TotalAmount', 'sum'),
    TotalQuantity=('Quantity', 'sum'),
    UniqueProducts=('StockCode', 'nunique'),
    ActiveDays=('InvoiceDate', lambda x: x.dt.date.nunique())
).reset_index()

print("Customer-level dataset created!")
print("Customers:", len(customer_features))
print("Features:", customer_features.shape[1])

customer_features.head()

In [ ]:
# Count future purchases for each customer
future_purchases = future.groupby('CustomerID')['InvoiceNo'].nunique()

# Add future purchase count to customer features
customer_features['FuturePurchases'] = (
    customer_features['CustomerID']
    .map(future_purchases)
    .fillna(0)
)

# Create churn target
# 1 = customer made no purchase in the next 90 days
# 0 = customer made at least one purchase
customer_features['Churn'] = (
    customer_features['FuturePurchases'] == 0
).astype(int)

print("Churn target created!")
print(customer_features['Churn'].value_counts())
print("\nChurn percentage:")
print(customer_features['Churn'].value_counts(normalize=True) * 100)

In [ ]:
# Remove future information from the ML features
model_data = customer_features.drop(columns=['FuturePurchases'])

print("Final ML dataset shape:", model_data.shape)
print("\nColumns:")
print(model_data.columns.tolist())

## 6. Inspect the prepared customer dataset

In [ ]:
# Basic statistics of the customer dataset
model_data.describe()

In [ ]:
# Check whether any missing values remain
print("Missing values:")
print(model_data.isnull().sum())

In [ ]:
# Check duplicate customer records
print("Duplicate rows:", model_data.duplicated().sum())

## 7. Exploratory Data Analysis (EDA)
The following visualizations inspect class distribution and how customer behavior differs between churned and retained customers.

In [ ]:
plt.figure(figsize=(7, 5))

sns.countplot(data=model_data, x='Churn')

plt.title('Customer Churn Distribution')
plt.xlabel('Churn (0 = Retained, 1 = Churned)')
plt.ylabel('Number of Customers')

plt.show()

In [ ]:
plt.figure(figsize=(8, 5))

sns.boxplot(data=model_data, x='Churn', y='Recency')

plt.title('Recency Distribution by Churn Status')
plt.xlabel('Churn (0 = Retained, 1 = Churned)')
plt.ylabel('Recency (Days)')

plt.show()

In [ ]:
plt.figure(figsize=(8, 5))

sns.boxplot(data=model_data, x='Churn', y='Frequency')

plt.title('Purchase Frequency by Churn Status')
plt.xlabel('Churn (0 = Retained, 1 = Churned)')
plt.ylabel('Number of Orders')

plt.show()

In [ ]:
plt.figure(figsize=(8, 5))

sns.boxplot(data=model_data, x='Churn', y='Monetary')

plt.title('Customer Spending by Churn Status')
plt.xlabel('Churn (0 = Retained, 1 = Churned)')
plt.ylabel('Total Spending')

plt.show()

In [ ]:
plt.figure(figsize=(10, 7))

correlation = model_data.drop(columns=['CustomerID']).corr()

sns.heatmap(
    correlation,
    annot=True,
    fmt='.2f',
    cmap='coolwarm',
    linewidths=0.5
)

plt.title('Correlation Heatmap of Customer Features')
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))

features = [
    'Recency',
    'Frequency',
    'Monetary',
    'TotalQuantity',
    'UniqueProducts',
    'ActiveDays'
]

for ax, feature in zip(axes.flatten(), features):
    sns.histplot(data=model_data, x=feature, kde=True, ax=ax)
    ax.set_title(f'Distribution of {feature}')

plt.tight_layout()
plt.show()

## 8. Prepare features and split the data
`CustomerID` is excluded from model features because it is an identifier rather than a behavioral measure. The data is split into 80% training and 20% testing data, with stratification to preserve the churn-class proportions.

In [ ]:
# Separate features and target

X = model_data.drop(columns=['CustomerID', 'Churn'])
y = model_data['Churn']

print("Features:")
print(X.columns.tolist())

print("\nFeature shape:", X.shape)
print("Target shape:", y.shape)

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

print("\nTraining churn distribution:")
print(y_train.value_counts())

print("\nTesting churn distribution:")
print(y_test.value_counts())

## 9. Model 1 — Logistic Regression
A StandardScaler is included in a pipeline because Logistic Regression is sensitive to feature scale.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

# Create a pipeline:
# 1. Standardize the features
# 2. Train Logistic Regression
logistic_model = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])

# Train the model
logistic_model.fit(X_train, y_train)

print("Logistic Regression trained successfully!")

In [ ]:
# Predict churn classes
y_pred = logistic_model.predict(X_test)

# Predict churn probabilities
y_prob = logistic_model.predict_proba(X_test)[:, 1]

print("Predictions generated successfully!")
print("Number of predictions:", len(y_pred))

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

# Calculate metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_prob)

print("MODEL EVALUATION RESULTS")
print("=" * 35)
print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1 Score : {f1:.4f}")
print(f"ROC-AUC  : {roc_auc:.4f}")

print("\nClassification Report:")
print(classification_report(
    y_test,
    y_pred,
    target_names=['Retained', 'Churned']
))

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

cm = confusion_matrix(y_test, y_pred)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=['Retained', 'Churned']
)

disp.plot()
plt.title('Confusion Matrix - Logistic Regression')
plt.show()

## 10. Model 2 — Random Forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=8,
    min_samples_split=10,
    random_state=42,
    class_weight='balanced'
)

rf_model.fit(X_train, y_train)

print("Random Forest trained successfully!")

In [ ]:
# Make predictions
rf_pred = rf_model.predict(X_test)
rf_prob = rf_model.predict_proba(X_test)[:, 1]

# Calculate metrics
rf_accuracy = accuracy_score(y_test, rf_pred)
rf_precision = precision_score(y_test, rf_pred)
rf_recall = recall_score(y_test, rf_pred)
rf_f1 = f1_score(y_test, rf_pred)
rf_roc_auc = roc_auc_score(y_test, rf_prob)

print("RANDOM FOREST EVALUATION RESULTS")
print("=" * 40)
print(f"Accuracy : {rf_accuracy:.4f}")
print(f"Precision: {rf_precision:.4f}")
print(f"Recall   : {rf_recall:.4f}")
print(f"F1 Score : {rf_f1:.4f}")
print(f"ROC-AUC  : {rf_roc_auc:.4f}")

print("\nClassification Report:")
print(classification_report(
    y_test,
    rf_pred,
    target_names=['Retained', 'Churned']
))

In [ ]:
# Random Forest feature importance

feature_importance = pd.DataFrame({
    'Feature': X_train.columns,
    'Importance': rf_model.feature_importances_
}).sort_values(by='Importance', ascending=False)

print(feature_importance)

plt.figure(figsize=(9, 5))

sns.barplot(
    data=feature_importance,
    x='Importance',
    y='Feature'
)

plt.title('Random Forest Feature Importance')
plt.xlabel('Importance')
plt.ylabel('Feature')

plt.show()

## 11. Model 3 — XGBoost
The following cell installs XGBoost in Google Colab if needed.

In [ ]:
!pip install -q xgboost

In [ ]:
from xgboost import XGBClassifier

xgb_model = XGBClassifier(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric='logloss',
    random_state=42
)

xgb_model.fit(X_train, y_train)

print("XGBoost trained successfully!")

In [ ]:
# Generate predictions
xgb_pred = xgb_model.predict(X_test)
xgb_prob = xgb_model.predict_proba(X_test)[:, 1]

# Calculate evaluation metrics
xgb_accuracy = accuracy_score(y_test, xgb_pred)
xgb_precision = precision_score(y_test, xgb_pred)
xgb_recall = recall_score(y_test, xgb_pred)
xgb_f1 = f1_score(y_test, xgb_pred)
xgb_roc_auc = roc_auc_score(y_test, xgb_prob)

# Display results
print("XGBOOST EVALUATION RESULTS")
print("=" * 40)
print(f"Accuracy : {xgb_accuracy:.4f}")
print(f"Precision: {xgb_precision:.4f}")
print(f"Recall   : {xgb_recall:.4f}")
print(f"F1 Score : {xgb_f1:.4f}")
print(f"ROC-AUC  : {xgb_roc_auc:.4f}")

print("\nClassification Report:")
print(classification_report(
    y_test,
    xgb_pred,
    target_names=['Retained', 'Churned']
))

## 12. Compare all models
The comparison below calculates metrics from the current test-set predictions. Higher scores are not automatically better for every metric; model selection should consider the project objective.

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)
import pandas as pd

# Generate predictions and probabilities
log_pred = logistic_model.predict(X_test)
log_prob = logistic_model.predict_proba(X_test)[:, 1]

rf_pred = rf_model.predict(X_test)
rf_prob = rf_model.predict_proba(X_test)[:, 1]

xgb_pred = xgb_model.predict(X_test)
xgb_prob = xgb_model.predict_proba(X_test)[:, 1]


# Function to calculate evaluation metrics
def evaluate_model(name, y_true, predictions, probabilities):
    return {
        'Model': name,
        'Accuracy': accuracy_score(y_true, predictions),
        'Precision': precision_score(y_true, predictions),
        'Recall': recall_score(y_true, predictions),
        'F1 Score': f1_score(y_true, predictions),
        'ROC-AUC': roc_auc_score(y_true, probabilities)
    }


# Build comparison table
comparison = pd.DataFrame([
    evaluate_model('Logistic Regression', y_test, log_pred, log_prob),
    evaluate_model('Random Forest', y_test, rf_pred, rf_prob),
    evaluate_model('XGBoost', y_test, xgb_pred, xgb_prob)
])

print("MODEL COMPARISON")
print("=" * 75)
print(comparison.round(4).to_string(index=False))

## 13. Confusion matrix for the selected model — Random Forest

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix

rf_cm = confusion_matrix(y_test, rf_pred)
rf_display = ConfusionMatrixDisplay(
    confusion_matrix=rf_cm,
    display_labels=["Retained", "Churned"]
)
rf_display.plot()
plt.title("Confusion Matrix — Random Forest")
plt.show()

## 14. ROC curve comparison

In [ ]:
from sklearn.metrics import RocCurveDisplay
from sklearn.metrics import roc_auc_score

plt.figure(figsize=(8, 6))
RocCurveDisplay.from_estimator(logistic_model, X_test, y_test, name="Logistic Regression")
RocCurveDisplay.from_estimator(rf_model, X_test, y_test, name="Random Forest")
RocCurveDisplay.from_estimator(xgb_model, X_test, y_test, name="XGBoost")
plt.title("ROC Curves — Model Comparison")
plt.show()

## 15. Results and model selection

The recorded evaluation results from the project run were:

| Model | Accuracy | Precision | Recall | F1-score | ROC-AUC |
|---|---:|---:|---:|---:|---:|
| Logistic Regression | 0.6717 | 0.6174 | 0.6598 | 0.6379 | 0.7349 |
| Random Forest | 0.6596 | 0.5876 | 0.7491 | 0.6586 | 0.7283 |
| XGBoost | 0.6491 | 0.5960 | 0.6186 | 0.6071 | 0.7129 |

Random Forest is selected for the deployed demonstration because it achieved the highest F1-score and recall among the three evaluated models. Logistic Regression achieved the highest accuracy and ROC-AUC. These results reflect this particular split of a historical dataset.

## 16. Save the selected model and feature order
Run this cell in Colab to recreate the artifacts used by the Streamlit app.

In [ ]:
import joblib
import os

# Create a folder for saved project files
os.makedirs('/content/project_artifacts', exist_ok=True)

# Save the selected model
joblib.dump(
    rf_model,
    '/content/project_artifacts/churn_model.joblib'
)

# Save the feature names in the correct order
joblib.dump(
    X_train.columns.tolist(),
    '/content/project_artifacts/feature_names.joblib'
)

print("Final model saved successfully!")
print("Feature names saved successfully!")

## 17. Conclusion and limitations

This project demonstrates a customer churn classification workflow from transaction cleaning through customer-level feature engineering, EDA, model comparison, and deployment preparation. Random Forest was selected for the Streamlit demonstration based on its F1-score and recall on the held-out test set.

### Limitations
- The dataset is historical and covers 2010–2011.
- The snapshot date and 90-day target window are fixed for this academic project.
- Results depend on the train-test split and the chosen features and hyperparameters.
- The model's probabilities are estimates, not guarantees about individual customers.
- Further validation on newer data and threshold tuning would be needed before real business use.